# 03 Report results

Review generated result summaries for available report models and show the command that regenerates the full numbered report figure set when all local checkpoints are present.

In [ ]:
from __future__ import annotations

import os
import sys
from pathlib import Path

os.environ.setdefault('TF_CPP_MIN_LOG_LEVEL', '2')

cwd = Path.cwd().resolve()
if (cwd / 'src').exists():
    project_root = cwd
elif (cwd.parent / 'src').exists():
    project_root = cwd.parent
else:
    raise RuntimeError('Run this notebook from the repository root or notebooks directory.')

os.chdir(project_root)
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

print(f'Project root: {project_root}')


In [ ]:
from IPython.display import Image, Markdown, display

from src.evaluation.comparison import format_comparison_markdown, load_run_summary


## Compare available report models

In [ ]:
REQUESTED_RUNS = (
    'baseline-model',
    'improved-model',
    'webcam-model',
)

RUNS = tuple(
    run for run in REQUESTED_RUNS
    if (project_root / 'artifacts' / run / 'evaluation.json').exists()
)
if not RUNS:
    raise FileNotFoundError('No evaluation artifacts found. Run `python -m src.evaluation.evaluate_run webcam-model` first.')

summaries = [load_run_summary(run) for run in RUNS]
display(Markdown(format_comparison_markdown(summaries)))

## Display the generated result summary

In [ ]:
summary_path = project_root / 'reports' / 'result-summary.md'
if summary_path.exists():
    display(Markdown(summary_path.read_text()))
else:
    command = 'python -m src.evaluation.report_summary ' + ' '.join(RUNS)
    print(f'No generated summary found. Generate it with: {command}')

## Regenerate report figures


In [ ]:
command = '''python -m src.evaluation.report_figures \
  baseline-model \
  improved-model \
  webcam-model'''
print(command)

figure_dir = project_root / 'reports' / 'report-figures'
figure_paths = sorted(
    figure_dir.glob('figure*.png'),
    key=lambda path: int(path.stem.removeprefix('figure')),
)

if len(RUNS) == 3 and len(figure_paths) == 11:
    for path in figure_paths:
        print(path.relative_to(project_root))
elif len(RUNS) == 3:
    print('All three models are evaluated locally; run the command above to regenerate report figures.')
else:
    print('Full report figure generation expects local baseline, improved, and webcam model artifacts.')

## Inspect selected report figures


In [ ]:
for figure_name in ('figure8.png', 'figure9.png', 'figure11.png'):
    path = figure_dir / figure_name
    if path.exists():
        display(Markdown(f'### {figure_name}'))
        display(Image(filename=str(path)))
    else:
        print(f'Missing {path.relative_to(project_root)}')